# invertibility_ma

> 对应代码：`EconometricsCode/code_in_notes/Chap.26/invertibility_ma.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.26`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.26")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们先生成一个模拟的时间轴和白噪声序列，然后构造两个 MA(1) 过程：y1 的 MA 系数为 0.5（满足可逆性条件），y2 的 MA 系数为 2（不满足可逆性条件）。两者使用完全相同的冲击 e，以便后文比较由观测序列反推冲击的效果。

In [ ]:
%%stata
clear
set obs 200
gen t=_n
tsset t
gen e=rnormal()
gen y1=e+0.5*L.e
gen y2=e+2*L.e

接下来我们从观测到的 y 序列反推冲击 e。对于可逆的 MA(1) 过程 y1（系数 0.5，根的模大于 1），我们可以把它写成无穷阶自回归表示 e_t = y_t - 0.5 y_{t-1} + 0.5^2 y_{t-2} - ...，这里截断到 5 阶滞后；由于系数 0.5^k 快速衰减，有限阶近似已经足够精确。

In [ ]:
%%stata
gen e1=y1-0.5*L.y1+0.5^2*L2.y1-0.5^3*L3.y1+0.5^4*L4.y1-0.5^5*L5.y1

对于不可逆的 MA(1) 过程 y2（系数 2），虽然其自回归表示不收敛，但我们仍可以利用 y_t 的''前向''展开反推 e：把 e 表示为未来 y 的加权和，即 e_t = 0.5 y_{t+1} - 0.5^2 y_{t+2} + ...（这里 y2=2e_{t-1}+e_t 的等价表达 e_t = y_t - 2 e_{t-1} 无法后向展开，但可以前向展开）。这说明不可逆 MA 过程的冲击仍然可以由数据恢复，只不过需要使用未来的观测值，而不是过去的观测值。

In [ ]:
%%stata
gen e2=0.5*F.y2-0.5^2*F2.y2+0.5^3*F3.y2-0.5^4*F4.y2+0.5^5*F5.y2

最后我们把真实冲击 e 与两种方法反推得到的 e1、e2 画在同一张图上，可以直观看到两者都能较好地还原真实冲击：可逆过程用历史信息即可，不可逆过程则需要未来信息。

In [ ]:
%%stata
tsline e e1 e2